# ✅ Validation of the Change Detection Model (Sobral, 2016–2023)

This notebook lets any user **reproduce the accuracy of the change detection component** reported in the paper
(Table 8) and the **benchmark against spectral index differencing**, using only the files in `Validation/data/`.

| File | Content |
|---|---|
| `sentinel2_Sobral_Belchior_t1.tif` / `_t2.tif` | Sentinel-2 L1C (COPERNICUS/S2_HARMONIZED), bands B2, B3, B4, B8, B11, B12 (reflectance), 10 m, EPSG:32724. T1 = 2016-09-04 (`20160904T131242_20160904T131242_T24MUB`), T2 = 2023-11-12 (`20231112T131239_20231112T131242_T24MUA`) |
| `sentinel1_Sobral_Belchior_t1.tif` / `_t2.tif` | Sentinel-1 VV/VH composites used as model input |
| `pred_Sobral_Belchior.tif` | U-Net output (band 1 = change probability 0–100), pre-trained weights of Hafner, Ban and Nascetti (2023), no fine-tuning |
| `ground_truth.tif` | Reference map of new buildings delineated by the authors (band 1: 255 = new building, 0 = no change; ≈1.12 m, EPSG:32724) |
| `mask_gain_NBAI_postclassification.tif` | Built-up gain from NBAI > −0.8 in T1 and T2 (post-classification alternative) |

**Expected results** (grid of the prediction, 81,792 pixels, 4,096 of change):

| Method | F1 | IoU | Precision | Recall | κ |
|---|---|---|---|---|---|
| U-Net, threshold 50 (model default) | 0.554 | 0.383 | 0.455 | 0.708 | 0.525 |
| U-Net, threshold 45 (best F1) | 0.554 | 0.383 | 0.440 | 0.747 | 0.524 |
| ΔNBAI, same detected area as the U-Net | 0.335 | 0.201 | 0.275 | 0.428 | 0.292 |
| NBAI post-classification (> −0.8) | 0.277 | 0.161 | 0.214 | 0.394 | 0.227 |
| ΔNDBI, Otsu threshold | 0.078 | 0.041 | 0.042 | 0.646 | −0.018 |

Metrics of the change class follow Hafner et al. (2023) and Corley et al. (2024). Confidence intervals use a spatial
block bootstrap (blocks of 32 × 32 px = 320 m), so neighbouring pixels are not treated as independent.

In [ ]:
#@title 1 — Setup {display-mode: "form"}
#@markdown Folder of the project on Google Drive (must contain `modulos/` and `Validation/data/`).
#@markdown If you only cloned the GitHub repository, choose **Clone from GitHub**.
source_mode = 'Google Drive'  #@param ['Google Drive', 'Clone from GitHub']
PROJECT_DIR = '/content/drive/MyDrive/Framework_Analise_Termica'  #@param {type:"string"}

import os, sys, subprocess
if source_mode == 'Google Drive':
    from google.colab import drive
    drive.mount('/content/drive')
else:
    PROJECT_DIR = '/content/ThermalAnalisys'
    if not os.path.isdir(PROJECT_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/JoaoVictorNH/ThermalAnalisys', PROJECT_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio'], check=True)
sys.path.insert(0, PROJECT_DIR)
DATA = os.path.join(PROJECT_DIR, 'Validation', 'data') + '/'
print('Data folder:', DATA)
print('\n'.join(sorted(os.listdir(DATA))))

In [ ]:
#@title 2 — Align the reference map to the prediction grid {display-mode: "form"}
#@markdown Nearest-neighbour resampling of the ≈1.12 m reference map onto the 10 m grid of the U-Net output.
import numpy as np
import rasterio
from modulos.cd_benchmark import align_reference, binary_change_metrics

with rasterio.open(DATA + 'pred_Sobral_Belchior.tif') as ds:
    prob = ds.read(1).astype(np.float32)
    grid = dict(like_transform=ds.transform, like_crs=ds.crs, like_shape=prob.shape)
ref = align_reference(DATA + 'ground_truth.tif', **grid)          # 1 change, 0 no change
valid = np.isfinite(ref)
y_ref = ref[valid].astype(bool)
print(f'Grid: {prob.shape} | reference pixels: {valid.sum():,} | change: {y_ref.sum():,} ({y_ref.mean():.2%})')
print('Expected: 81,792 pixels, 4,096 of change (5.01%).')

In [ ]:
#@title 3 — Accuracy of the U-Net (Table 8) and threshold sensitivity {display-mode: "form"}
import pandas as pd
rows = []
for t in range(5, 100, 5):
    m = binary_change_metrics(prob[valid] >= t, y_ref)
    rows.append({'threshold': t, **{k: m[k] for k in ('F1', 'IoU', 'precision', 'recall', 'kappa', 'TP', 'FP', 'TN', 'FN')}})
sweep = pd.DataFrame(rows).set_index('threshold')
display(sweep.round(4))
for t in (50, 45):
    print(f"threshold {t}: " + ', '.join(f"{k} = {sweep.loc[t, k]:.3f}" for k in ('F1', 'IoU', 'precision', 'recall', 'kappa')))
print('\nNote: 45 is the threshold with the best F1 on this same area (it is what cell 6.4 of the pipeline uses);')
print('the model default (50) gives the same F1, so the reported accuracy does not depend on that choice.')

In [ ]:
#@title 4 — Concordance map {display-mode: "form"}
threshold_map = 50  #@param {type:"slider", min:5, max:95, step:5}
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
pred = prob >= threshold_map
r = ref.astype(bool)
cat = np.select([r & pred, ~r & pred, r & ~pred], [1, 2, 3], 0).astype(float)
cat[~valid] = np.nan
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(np.where(valid, ref, np.nan), cmap=ListedColormap(['#f0f0f0', '#222222']), interpolation='nearest')
ax[0].set_title('Reference map (new buildings in black)')
ax[1].imshow(cat, cmap=ListedColormap(['#f0f0f0', '#1a9850', '#d73027', '#4575b4']), vmin=0, vmax=3, interpolation='nearest')
ax[1].set_title(f'U-Net, threshold {threshold_map}')
for a in ax: a.set_xticks([]); a.set_yticks([])
fig.legend(handles=[Patch(color='#1a9850', label='TP'), Patch(color='#d73027', label='FP (false alarm)'),
                    Patch(color='#4575b4', label='FN (missed)')], loc='lower center', ncol=3, frameon=False)
plt.show()

In [ ]:
#@title 5 — Benchmark: U-Net vs spectral index differencing {display-mode: "form"}
#@markdown Same Sentinel-2 images and same reference map. For each index: **Otsu** threshold (no reference needed)
#@markdown and the threshold that flags the **same area** as the U-Net. ΔNDBI is the basis of the CDI of Silva and Torres (2021).
unet_threshold = 50  #@param {type:"slider", min:5, max:95, step:5}
n_boot = 1000  #@param {type:"slider", min:200, max:2000, step:100}
from modulos.cd_benchmark import benchmark_change_detection, print_change_benchmark, plot_change_benchmark
res = benchmark_change_detection(DATA, 'Sobral_Belchior', DATA + 'ground_truth.tif',
                                 unet_threshold=unet_threshold, n_boot=n_boot)
print_change_benchmark(res)
fig = plot_change_benchmark(res); plt.show()

with rasterio.open(DATA + 'mask_gain_NBAI_postclassification.tif') as ds:
    nb = ds.read(1)
m = binary_change_metrics(nb[valid] > 0, y_ref)
print('\nNBAI post-classification (NBAI > -0.8 in T1 and T2): ' +
      ', '.join(f"{k} = {m[k]:.3f}" for k in ('F1', 'IoU', 'precision', 'recall', 'kappa')))

In [ ]:
#@title 6 — (Optional) Re-run the U-Net from the Sentinel-1/2 inputs {display-mode: "form"}
#@markdown Regenerates `pred_Sobral_Belchior.tif` from the inputs in `Validation/data/` with the pre-trained weights
#@markdown (`mmcr_train100.pt`, see the main README for the download) and compares it with the supplied file.
#@markdown The new prediction is written to `Validation/rerun/` (the supplied file is not overwritten).
run_model = False  #@param {type:"boolean"}
folder_model = '/content/drive/MyDrive/Framework_Analise_Termica/Modelo_Rede_Neural_Change_Detection/'  #@param {type:"string"}
if run_model:
    import shutil
    from modulos.change_detection import run_inference
    rerun = os.path.join(PROJECT_DIR, 'Validation', 'rerun') + '/'
    os.makedirs(rerun, exist_ok=True)
    for f in ('sentinel1_Sobral_Belchior_t1.tif', 'sentinel1_Sobral_Belchior_t2.tif',
              'sentinel2_Sobral_Belchior_t1.tif', 'sentinel2_Sobral_Belchior_t2.tif'):
        shutil.copy(DATA + f, rerun + f)
    new_pred = run_inference(rerun, 'Sobral_Belchior', folder_model, model_name='mmcr_train100')
    with rasterio.open(new_pred) as ds:
        p2 = ds.read(1).astype(np.float32)
    print(f'Mean absolute difference to the supplied prediction: {np.abs(p2 - prob).mean():.3f} (probability 0–100)')
    m = binary_change_metrics(p2[valid] >= 50, y_ref)
    print('Re-run, threshold 50: ' + ', '.join(f"{k} = {m[k]:.3f}" for k in ('F1', 'IoU', 'precision', 'recall', 'kappa')))